# Building LLM

## Goal

This notebook is a hands-on journey to build a language model from scratch.

Each version introduces one new concept, allowing the model to evolve step by step while practicing language-model development.

---

## Version 13

Version 12 added explicit training state, last and best checkpoints, simulated interruption, restoration and resume to the WikiText/BPE language model.

Version 13 keeps the same dataset, tokenizer, token sequences, decoder-only Transformer, optimization settings and checkpoint pipeline.

The new concept is **held-out evaluation and controlled generation**.

The progression is:

restored best model  
→ held-out test cross-entropy  
→ perplexity and token accuracy  
→ final-position logits  
→ temperature, top-k and top-p controls  
→ reproducible autoregressive sampling

The test split remains isolated until validation has selected and restored the best checkpoint.

Version 13 therefore extends the post-training analysis of Version 12 without changing the language-model architecture or training budget.

## 1. Imports and Configuration

Version 13 preserves the Version 12 experiment:

- 10,000 training rows, 1,000 validation rows and 1,000 test rows;
- a BPE vocabulary of 2,048 tokens;
- contexts of 16 tokens;
- 12,000 training sequences, 2,000 validation sequences and 2,000 test sequences;
- batch size 32;
- learning rate 1.0;
- the same two-block decoder-only Transformer;
- interruption at epoch 10 and a total budget of 20 epochs.

Training examples continue to use an epoch-specific shuffle seed so checkpoint restoration resumes with the correct next epoch.

The new work begins only after the best validation checkpoint has been restored: Version 13 adds held-out metrics and explicit decoding controls while keeping the rest of the experiment unchanged.

In [1]:
import os
import shutil
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["RAYON_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import tensorflow as tf

from tokenizers import Tokenizer
from tokenizers.decoders import BPEDecoder
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer

In [2]:
SEED = 42

TRAIN_ROWS = 10000
VALIDATION_ROWS = 1000
TEST_ROWS = 1000

VOCABULARY_SIZE = 2048
CONTEXT_LENGTH = 16

MAX_TRAIN_SEQUENCES = 12000
MAX_VALIDATION_SEQUENCES = 2000
MAX_TEST_SEQUENCES = 2000

BATCH_SIZE = 32
LEARNING_RATE = 1.0

INTERRUPT_AFTER_EPOCH = 10
TOTAL_EPOCHS = 20

EMBEDDING_DIM = 8
ATTENTION_DIM = 16
FEED_FORWARD_DIM = 16
NUM_BLOCKS = 2
LAYER_NORM_EPSILON = 1e-5

tf.keras.utils.set_random_seed(SEED)

In [3]:
DATASET_DIRECTORY = "/kaggle/input/datasets/lucalullo/wiki-text-103-raw-v1"

train_files = [
    os.path.join(DATASET_DIRECTORY, "train-00000-of-00002.parquet"),
    os.path.join(DATASET_DIRECTORY, "train-00001-of-00002.parquet"),
]

validation_file = os.path.join(DATASET_DIRECTORY, "validation-00000-of-00001.parquet")
test_file = os.path.join(DATASET_DIRECTORY, "test-00000-of-00001.parquet")

for path in train_files + [validation_file, test_file]:
    if not os.path.exists(path):
        raise FileNotFoundError(path)

print("Dataset files found:")
for path in train_files + [validation_file, test_file]:
    print(os.path.basename(path))

Dataset files found:
train-00000-of-00002.parquet
train-00001-of-00002.parquet
validation-00000-of-00001.parquet
test-00000-of-00001.parquet


## 2. Public Dataset

Version 13 keeps the **WikiText-103 Raw v1** dataset introduced in Version 11.

The dataset is attached to the Kaggle notebook as an input and contains the original training, validation and test splits in Parquet format.

The files are:

- two Parquet shards for the training split;
- one Parquet file for the validation split;
- one Parquet file for the test split.

Each file contains a single `text` column.

The original split boundaries remain unchanged. Version 13 does not create a new random train/validation/test split.

To preserve direct comparability with Versions 11 and 12, the same reproducible subset is selected independently from each official split:

- 10,000 training rows;
- 1,000 validation rows;
- 1,000 test rows.

Empty rows are removed before sampling because WikiText contains blank entries that represent document and section boundaries rather than useful standalone training text.

The dataset path is validated explicitly before the files are read. If the external Kaggle input is unavailable, the notebook stops rather than downloading or substituting another corpus.

The dataset language is English. Source, provenance and licensing information remain those documented for the attached `wikitext-103-raw-v1` dataset.

In [4]:
train_data = pd.concat(
    [pd.read_parquet(path, columns=["text"]) for path in train_files],
    ignore_index=True,
)

validation_data = pd.read_parquet(validation_file, columns=["text"])
test_data = pd.read_parquet(test_file, columns=["text"])

print("Original rows:")
print("Train:", len(train_data))
print("Validation:", len(validation_data))
print("Test:", len(test_data))

Original rows:
Train: 1801350
Validation: 3760
Test: 4358


In [5]:
def prepare_text_subset(data, number_of_rows, seed):
    cleaned = data.dropna(subset=["text"]).copy()
    cleaned = cleaned[cleaned["text"].str.strip().str.len() > 0]
    return cleaned.sample(n=min(number_of_rows, len(cleaned)), random_state=seed).reset_index(drop=True)


train_data = prepare_text_subset(train_data, TRAIN_ROWS, SEED)
validation_data = prepare_text_subset(validation_data, VALIDATION_ROWS, SEED)
test_data = prepare_text_subset(test_data, TEST_ROWS, SEED)

print("Experiment rows:")
print("Train:", len(train_data))
print("Validation:", len(validation_data))
print("Test:", len(test_data))

print("\nExample training text:")
print(train_data.loc[0, "text"])

Experiment rows:
Train: 10000
Validation: 1000
Test: 1000

Example training text:
 In cooperation with various nonprofit organizations , since 2008 the company has organized the THSRC Smile Program , which offers minority groups and families the chance to experience the comfort and convenience of high @-@ speed rail rides . THSRC Smile Program gave its first ride on 23 January 2008 to children , social workers and parents from the Fushan branch of the Taiwan Fund for Children and Families . As of the end of 2011 , 12 @,@ 231 people in total , comprising 229 underprivileged groups and families , have participated in the THSRC Smile Program . 



## 3. Subword Tokenizer

Version 13 keeps the **Byte Pair Encoding (BPE)** tokenizer introduced in Version 11.

BPE starts from small text units and repeatedly merges frequent neighboring units to build a vocabulary of reusable subword tokens.

The tokenizer is trained only on the selected training subset. Validation and test text are never used to construct the vocabulary.

The tokenizer configuration remains unchanged:

- a maximum vocabulary size of 2,048 tokens;
- `[UNK]` as the special unknown token;
- whitespace pre-tokenization;
- a minimum token frequency of 2;
- `</w>` as the end-of-word suffix.

Keeping the same tokenizer procedure ensures that Version 13 changes evaluation and decoding rather than the language representation.

In [6]:
tokenizer = Tokenizer(BPE(
    unk_token="[UNK]",
    end_of_word_suffix="</w>"
))

tokenizer.pre_tokenizer = Whitespace()
tokenizer.decoder = BPEDecoder(suffix="</w>")

trainer = BpeTrainer(
    vocab_size=VOCABULARY_SIZE,
    min_frequency=2,
    special_tokens=["[UNK]"],
    end_of_word_suffix="</w>"
)

tokenizer.train_from_iterator(train_data["text"], trainer=trainer)

vocabulary_size = tokenizer.get_vocab_size()

print("Vocabulary size:", vocabulary_size)




Vocabulary size: 2048


### Inspecting the Learned Tokens

The tokenizer can now represent text using variable-length subword units instead of individual characters.

Frequent words or word fragments may become single tokens, while less frequent words can be decomposed into multiple smaller tokens.

In [7]:
example_text = train_data.loc[0, "text"]
encoded_example = tokenizer.encode(example_text)

print("Text:")
print(example_text)

print("\nTokens:")
print(encoded_example.tokens[:30])

print("\nToken IDs:")
print(encoded_example.ids[:30])

print("\nNumber of characters:", len(example_text))
print("Number of tokens:", len(encoded_example.ids))

Text:
 In cooperation with various nonprofit organizations , since 2008 the company has organized the THSRC Smile Program , which offers minority groups and families the chance to experience the comfort and convenience of high @-@ speed rail rides . THSRC Smile Program gave its first ride on 23 January 2008 to children , social workers and parents from the Fushan branch of the Taiwan Fund for Children and Families . As of the end of 2011 , 12 @,@ 231 people in total , comprising 229 underprivileged groups and families , have participated in the THSRC Smile Program . 


Tokens:
['In</w>', 'co', 'oper', 'ation</w>', 'with</w>', 'vari', 'ous</w>', 'n', 'on', 'pro', 'f', 'it</w>', 'org', 'ani', 'z', 'ations</w>', ',</w>', 'since</w>', '200', '8</w>', 'the</w>', 'comp', 'any</w>', 'has</w>', 'org', 'ani', 'zed</w>', 'the</w>', 'T', 'H']

Token IDs:
[1146, 1228, 1730, 1103, 1066, 1919, 1272, 78, 999, 1082, 70, 1102, 1875, 1735, 90, 1434, 854, 1927, 1196, 676, 991, 1452, 1389, 1260, 1875, 173

## 4. Token Sequences

The decoder-only Transformer still learns next-token prediction at every context position.

Each WikiText row is encoded independently with the trained BPE tokenizer.

Token sequences are divided into fixed-length windows of `CONTEXT_LENGTH + 1` tokens. The first `CONTEXT_LENGTH` tokens form the input and the same sequence shifted by one position forms the target.

For example:

input tokens  → `[t0, t1, t2, ..., t15]`  
target tokens → `[t1, t2, t3, ..., t16]`

Unlike the character-level examples used in earlier versions, the windows now contain subword token identifiers.

Windows are created independently inside each text row so examples never cross row boundaries. A stride equal to `CONTEXT_LENGTH` avoids excessive overlap and keeps the experiment compact.

In [8]:
def create_token_sequences(texts, tokenizer, context_length):
    inputs, targets = [], []

    for text in texts:
        token_ids = tokenizer.encode(text).ids

        for start in range(0, len(token_ids) - context_length, context_length):
            sequence = token_ids[start:start + context_length + 1]

            if len(sequence) == context_length + 1:
                inputs.append(sequence[:-1])
                targets.append(sequence[1:])

    return tf.constant(inputs, dtype=tf.int32), tf.constant(targets, dtype=tf.int32)

In [9]:
train_inputs, train_targets = create_token_sequences(train_data["text"], tokenizer, CONTEXT_LENGTH)
validation_inputs, validation_targets = create_token_sequences(validation_data["text"], tokenizer, CONTEXT_LENGTH)
test_inputs, test_targets = create_token_sequences(test_data["text"], tokenizer, CONTEXT_LENGTH)

print("Train:", train_inputs.shape, train_targets.shape)
print("Validation:", validation_inputs.shape, validation_targets.shape)
print("Test:", test_inputs.shape, test_targets.shape)

print("\nFirst input IDs:")
print(train_inputs[0].numpy())

print("\nFirst target IDs:")
print(train_targets[0].numpy())

print("\nDecoded input:")
print(tokenizer.decode(train_inputs[0].numpy()))

Train: (96893, 16) (96893, 16)
Validation: (9764, 16) (9764, 16)
Test: (9572, 16) (9572, 16)

First input IDs:
[1146 1228 1730 1103 1066 1919 1272   78  999 1082   70 1102 1875 1735
   90 1434]

First target IDs:
[1228 1730 1103 1066 1919 1272   78  999 1082   70 1102 1875 1735   90
 1434  854]

Decoded input:
In cooperation with various nonprofit organizations


In [10]:
train_inputs = train_inputs[:MAX_TRAIN_SEQUENCES]
train_targets = train_targets[:MAX_TRAIN_SEQUENCES]

validation_inputs = validation_inputs[:MAX_VALIDATION_SEQUENCES]
validation_targets = validation_targets[:MAX_VALIDATION_SEQUENCES]

test_inputs = test_inputs[:MAX_TEST_SEQUENCES]
test_targets = test_targets[:MAX_TEST_SEQUENCES]

## 5. Decoder-Only Transformer

The decoder-only Transformer remains mathematically unchanged from Versions 11 and 12.

Token embeddings, positional embeddings, two Transformer blocks and the final vocabulary projection keep the same dimensions and forward calculations.

Each Transformer block still contains:

- query, key and value projections;
- causal scaled dot-product self-attention;
- an attention output projection;
- a residual connection followed by LayerNorm;
- a position-wise feed-forward network;
- a second residual connection followed by LayerNorm.

Two independent Transformer blocks are stacked sequentially.

The model continues to inherit from `tf.Module` so the existing TensorFlow variables remain trackable by the checkpoint pipeline introduced in Version 12. This does not change the model mathematics or parameter count.

In [11]:
def layer_normalize(values, scale, shift):
    mean = tf.reduce_mean(values, axis=-1, keepdims=True)
    variance = tf.reduce_mean(tf.square(values - mean), axis=-1, keepdims=True)
    normalized = (values - mean) / tf.sqrt(variance + LAYER_NORM_EPSILON)
    return normalized * scale + shift


class TransformerBlock(tf.Module):
    def __init__(self, random_generator, name):
        super().__init__(name=name)

        self.query_weights = tf.Variable(random_generator.normal((EMBEDDING_DIM, ATTENTION_DIM), stddev=0.02), name="query_weights")
        self.key_weights = tf.Variable(random_generator.normal((EMBEDDING_DIM, ATTENTION_DIM), stddev=0.02), name="key_weights")
        self.value_weights = tf.Variable(random_generator.normal((EMBEDDING_DIM, ATTENTION_DIM), stddev=0.02), name="value_weights")
        self.attention_output_weights = tf.Variable(random_generator.normal((ATTENTION_DIM, EMBEDDING_DIM), stddev=0.02), name="attention_output_weights")

        self.layer_norm_1_scale = tf.Variable(tf.ones((EMBEDDING_DIM,)), name="layer_norm_1_scale")
        self.layer_norm_1_shift = tf.Variable(tf.zeros((EMBEDDING_DIM,)), name="layer_norm_1_shift")

        self.feed_forward_input_weights = tf.Variable(random_generator.normal((EMBEDDING_DIM, FEED_FORWARD_DIM), stddev=0.02), name="feed_forward_input_weights")
        self.feed_forward_output_weights = tf.Variable(random_generator.normal((FEED_FORWARD_DIM, EMBEDDING_DIM), stddev=0.02), name="feed_forward_output_weights")

        self.layer_norm_2_scale = tf.Variable(tf.ones((EMBEDDING_DIM,)), name="layer_norm_2_scale")
        self.layer_norm_2_shift = tf.Variable(tf.zeros((EMBEDDING_DIM,)), name="layer_norm_2_shift")

    @property
    def trainable_variables(self):
        return [
            self.query_weights, self.key_weights, self.value_weights,
            self.attention_output_weights,
            self.layer_norm_1_scale, self.layer_norm_1_shift,
            self.feed_forward_input_weights, self.feed_forward_output_weights,
            self.layer_norm_2_scale, self.layer_norm_2_shift,
        ]

    def __call__(self, values):
        queries = tf.matmul(values, self.query_weights)
        keys = tf.matmul(values, self.key_weights)
        values_projected = tf.matmul(values, self.value_weights)

        scores = tf.matmul(queries, keys, transpose_b=True)
        scores = scores / tf.sqrt(tf.cast(ATTENTION_DIM, tf.float32))

        sequence_length = tf.shape(scores)[-1]
        causal_mask = tf.linalg.band_part(tf.ones((sequence_length, sequence_length)), -1, 0)
        masked_scores = tf.where(causal_mask == 1, scores, tf.constant(-1e9, dtype=tf.float32))

        attention_weights = tf.nn.softmax(masked_scores, axis=-1)
        attention_output = tf.matmul(attention_weights, values_projected)
        attention_output = tf.matmul(attention_output, self.attention_output_weights)

        values = layer_normalize(values + attention_output, self.layer_norm_1_scale, self.layer_norm_1_shift)

        feed_forward = tf.nn.relu(tf.matmul(values, self.feed_forward_input_weights))
        feed_forward = tf.matmul(feed_forward, self.feed_forward_output_weights)

        output = layer_normalize(values + feed_forward, self.layer_norm_2_scale, self.layer_norm_2_shift)

        return output, attention_weights


class DecoderOnlyLanguageModel(tf.Module):
    def __init__(self, vocabulary_size, seed=42, name="decoder_only_language_model"):
        super().__init__(name=name)
        random_generator = tf.random.Generator.from_seed(seed)

        self.token_embeddings = tf.Variable(random_generator.normal((vocabulary_size, EMBEDDING_DIM), stddev=0.02), name="token_embeddings")
        self.positional_embeddings = tf.Variable(random_generator.normal((CONTEXT_LENGTH, EMBEDDING_DIM), stddev=0.02), name="positional_embeddings")

        self.blocks = [
            TransformerBlock(random_generator, name=f"transformer_block_{index + 1}")
            for index in range(NUM_BLOCKS)
        ]

        self.output_weights = tf.Variable(random_generator.normal((EMBEDDING_DIM, vocabulary_size), stddev=0.02), name="output_weights")

    @property
    def trainable_variables(self):
        variables = [self.token_embeddings, self.positional_embeddings]

        for block in self.blocks:
            variables.extend(block.trainable_variables)

        variables.append(self.output_weights)
        return variables

    def __call__(self, input_ids):
        token_values = tf.gather(self.token_embeddings, input_ids)
        values = token_values + self.positional_embeddings
        attention_weights = []

        for block in self.blocks:
            values, block_attention = block(values)
            attention_weights.append(block_attention)

        logits = tf.matmul(values, self.output_weights)
        return values, attention_weights, logits

In [12]:
vocabulary_size = tokenizer.get_vocab_size()

model = DecoderOnlyLanguageModel(vocabulary_size=vocabulary_size, seed=SEED)

print("Tokenizer vocabulary size:", vocabulary_size)
print("Token embedding shape:", model.token_embeddings.shape)

Tokenizer vocabulary size: 2048
Token embedding shape: (2048, 8)


### Model Structure and Trainable Parameters

The decoder-only Transformer operates on BPE token sequences with a vocabulary of 2,048 tokens and a context length of 16.

Two independent causal Transformer blocks process token and positional embeddings before the final projection produces logits over the complete vocabulary.

Inspecting tensor shapes and parameter count confirms that Version 13 preserves the same 34,496-parameter model used in Versions 11 and 12.

In [13]:
example_input = train_inputs[:1]

model_output, attention_weights, logits = model(example_input)

total_parameters = sum(int(np.prod(variable.shape)) for variable in model.trainable_variables)

print("Vocabulary size:", vocabulary_size)
print("Number of Transformer blocks:", len(model.blocks))
print("Number of trainable variables:", len(model.trainable_variables))

print("\nInput shape:", example_input.shape)
print("Model output shape:", model_output.shape)
print("Logits shape:", logits.shape)

for index, block_attention in enumerate(attention_weights, start=1):
    print(f"Block {index} attention shape:", block_attention.shape)

print("\nTotal trainable parameters:", total_parameters)

Vocabulary size: 2048
Number of Transformer blocks: 2
Number of trainable variables: 23

Input shape: (1, 16)
Model output shape: (1, 16, 8)
Logits shape: (1, 16, 2048)
Block 1 attention shape: (1, 16, 16)
Block 2 attention shape: (1, 16, 16)

Total trainable parameters: 34496


## 6. Reusable Training Pipeline

Version 13 preserves the reusable training pipeline introduced in Version 12.

The next-token objective, sparse categorical cross-entropy, manual gradient-descent update, batch size, learning rate and epoch-specific deterministic shuffling remain unchanged.

`make_dataset` creates mini-batches, `evaluate_model` measures a dataset without parameter updates, and `train_one_epoch` performs one training epoch.

The checkpoint-aware training loop later coordinates validation and checkpoint decisions while keeping the training mathematics identical to Version 12.

In [14]:
def make_dataset(inputs, targets, batch_size, shuffle_seed=None):
    dataset = tf.data.Dataset.from_tensor_slices((inputs, targets))

    if shuffle_seed is not None:
        dataset = dataset.shuffle(buffer_size=len(inputs), seed=shuffle_seed, reshuffle_each_iteration=False)

    return dataset.batch(batch_size)


validation_dataset = make_dataset(validation_inputs, validation_targets, BATCH_SIZE)
test_dataset = make_dataset(test_inputs, test_targets, BATCH_SIZE)

print("Training sequences:", len(train_inputs))
print("Validation sequences:", len(validation_inputs))
print("Test sequences:", len(test_inputs))
print("Batches per training epoch:", int(np.ceil(len(train_inputs) / BATCH_SIZE)))

Training sequences: 12000
Validation sequences: 2000
Test sequences: 2000
Batches per training epoch: 375


In [15]:
def compute_batch_loss(model, inputs, targets):
    _, _, logits = model(inputs)

    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=targets, logits=logits)

    return tf.reduce_mean(losses)

In [16]:
def evaluate_model(model, dataset):
    total_loss = 0.0
    total_tokens = 0

    for batch_inputs, batch_targets in dataset:
        batch_loss = compute_batch_loss(model, batch_inputs, batch_targets)
        batch_tokens = int(tf.size(batch_targets))

        total_loss += float(batch_loss) * batch_tokens
        total_tokens += batch_tokens

    return total_loss / total_tokens

### Gradient Descent

Each mini-batch performs one gradient-descent update.

TensorFlow's `GradientTape` records the operations used to compute the batch loss and obtains the gradient of that loss with respect to every trainable variable.

The parameters are then updated explicitly using the configured learning rate.

The mean training loss is accumulated from the mini-batches processed during each epoch, while validation loss is computed separately without modifying the model parameters.

Token embedding lookups can produce sparse `IndexedSlices` gradients. These are updated with `scatter_sub`, while dense gradients use the standard `assign_sub` update.

In [17]:
def apply_gradient(variable, gradient, learning_rate):
    if gradient is None:
        return

    if isinstance(gradient, tf.IndexedSlices):
        update = tf.IndexedSlices(
            values=learning_rate * gradient.values,
            indices=gradient.indices,
            dense_shape=gradient.dense_shape
        )
        variable.scatter_sub(update)

    else:
        variable.assign_sub(learning_rate * gradient)

In [18]:
def train_one_epoch(model, inputs, targets, epoch, learning_rate):
    training_dataset = make_dataset(
        inputs,
        targets,
        BATCH_SIZE,
        shuffle_seed=SEED + epoch,
    )

    total_loss = 0.0
    total_tokens = 0

    for batch_inputs, batch_targets in training_dataset:
        with tf.GradientTape() as tape:
            batch_loss = compute_batch_loss(model, batch_inputs, batch_targets)

        gradients = tape.gradient(batch_loss, model.trainable_variables)

        for variable, gradient in zip(model.trainable_variables, gradients):
            apply_gradient(variable, gradient, learning_rate)

        batch_tokens = int(tf.size(batch_targets))
        total_loss += float(batch_loss) * batch_tokens
        total_tokens += batch_tokens

    return total_loss / total_tokens

## 7. Training State and Checkpoints

Version 13 preserves the minimum checkpoint state introduced in Version 12.

Three values are tracked:

- `completed_epoch`, the most recently completed epoch;
- `best_epoch`, the epoch with the lowest validation loss;
- `best_validation_loss`, the lowest validation loss observed so far.

Two checkpoints keep their separate roles:

- the **last checkpoint** stores the current model and state required to resume;
- the **best checkpoint** stores the model selected by validation loss.

The runtime directory starts clean for each notebook execution. Training stops first at epoch 10, restores the last checkpoint into a fresh model, resumes through epoch 20, and finally restores the best checkpoint into `final_model`.

In [19]:
RUNTIME_DIRECTORY = Path("/kaggle/working/runtime-v13")

if RUNTIME_DIRECTORY.exists():
    shutil.rmtree(RUNTIME_DIRECTORY)

RUNTIME_DIRECTORY.mkdir(parents=True, exist_ok=True)

print("Runtime checkpoints:", RUNTIME_DIRECTORY)

Runtime checkpoints: /kaggle/working/runtime-v13


In [20]:
def build_checkpoint_managers(
    model,
    completed_epoch,
    best_epoch,
    best_validation_loss,
    runtime_directory,
):
    last_checkpoint = tf.train.Checkpoint(
        model=model,
        completed_epoch=completed_epoch,
        best_epoch=best_epoch,
        best_validation_loss=best_validation_loss,
    )

    best_checkpoint = tf.train.Checkpoint(
        model=model,
        best_epoch=best_epoch,
        best_validation_loss=best_validation_loss,
    )

    last_manager = tf.train.CheckpointManager(
        last_checkpoint,
        str(runtime_directory / "last"),
        max_to_keep=1,
    )

    best_manager = tf.train.CheckpointManager(
        best_checkpoint,
        str(runtime_directory / "best"),
        max_to_keep=1,
    )

    return last_checkpoint, best_checkpoint, last_manager, best_manager

In [21]:
def run_training_pipeline(
    model,
    completed_epoch,
    best_epoch,
    best_validation_loss,
    train_inputs,
    train_targets,
    validation_dataset,
    final_epoch,
    learning_rate,
    last_manager,
    best_manager,
):
    start_epoch = int(completed_epoch.numpy()) + 1

    for epoch in range(start_epoch, final_epoch + 1):
        training_loss = train_one_epoch(
            model,
            train_inputs,
            train_targets,
            epoch,
            learning_rate,
        )

        validation_loss = evaluate_model(model, validation_dataset)

        completed_epoch.assign(epoch)

        if validation_loss < float(best_validation_loss.numpy()):
            best_validation_loss.assign(validation_loss)
            best_epoch.assign(epoch)
            best_manager.save(checkpoint_number=epoch)

        last_manager.save(checkpoint_number=epoch)

        if epoch == 1 or epoch % 10 == 0 or epoch == final_epoch:
            print(
                f"Epoch {epoch:3d} | "
                f"Train loss: {training_loss:.6f} | "
                f"Validation loss: {validation_loss:.6f}"
            )

In [22]:
completed_epoch = tf.Variable(0, dtype=tf.int64, trainable=False)
best_epoch = tf.Variable(0, dtype=tf.int64, trainable=False)
best_validation_loss = tf.Variable(np.inf, dtype=tf.float32, trainable=False)

last_checkpoint, best_checkpoint, last_manager, best_manager = build_checkpoint_managers(
    model,
    completed_epoch,
    best_epoch,
    best_validation_loss,
    RUNTIME_DIRECTORY,
)

initial_train_dataset = make_dataset(train_inputs, train_targets, BATCH_SIZE)

initial_train_loss = evaluate_model(model, initial_train_dataset)
initial_validation_loss = evaluate_model(model, validation_dataset)
random_baseline = float(np.log(vocabulary_size))

print(f"Initial training loss: {initial_train_loss:.6f}")
print(f"Initial validation loss: {initial_validation_loss:.6f}")
print(f"Random baseline: {random_baseline:.6f}")

Initial training loss: 7.626188
Initial validation loss: 7.625834
Random baseline: 7.624619


## 8. Simulated Interruption, Restoration and Resume

The Version 12 checkpoint demonstration is preserved so the new Version 13 evaluation is tied to the same resumable training process.

Training first reaches epoch 10 and writes last and best checkpoints.

A fresh model and fresh state variables then restore the last checkpoint. Because `completed_epoch` is restored as 10, training resumes from epoch 11 and continues through epoch 20.

Validation continues to update the best checkpoint whenever a lower validation loss is observed.

In [23]:
run_training_pipeline(
    model,
    completed_epoch,
    best_epoch,
    best_validation_loss,
    train_inputs,
    train_targets,
    validation_dataset,
    INTERRUPT_AFTER_EPOCH,
    LEARNING_RATE,
    last_manager,
    best_manager,
)

print("\nInterrupted after epoch:", int(completed_epoch.numpy()))
print("Last checkpoint:", last_manager.latest_checkpoint)
print("Best checkpoint:", best_manager.latest_checkpoint)

Epoch   1 | Train loss: 5.987667 | Validation loss: 5.512638
Epoch  10 | Train loss: 5.083346 | Validation loss: 5.131816

Interrupted after epoch: 10
Last checkpoint: /kaggle/working/runtime-v13/last/ckpt-10
Best checkpoint: /kaggle/working/runtime-v13/best/ckpt-9


In [24]:
resumed_model = DecoderOnlyLanguageModel(vocabulary_size=vocabulary_size, seed=SEED)
_ = resumed_model(example_input)

resumed_completed_epoch = tf.Variable(0, dtype=tf.int64, trainable=False)
resumed_best_epoch = tf.Variable(0, dtype=tf.int64, trainable=False)
resumed_best_validation_loss = tf.Variable(np.inf, dtype=tf.float32, trainable=False)

(
    resumed_last_checkpoint,
    resumed_best_checkpoint,
    resumed_last_manager,
    resumed_best_manager,
) = build_checkpoint_managers(
    resumed_model,
    resumed_completed_epoch,
    resumed_best_epoch,
    resumed_best_validation_loss,
    RUNTIME_DIRECTORY,
)

if resumed_last_manager.latest_checkpoint is None:
    raise RuntimeError("Last checkpoint not found.")

resumed_last_checkpoint.restore(resumed_last_manager.latest_checkpoint).assert_existing_objects_matched()

restored_epoch = int(resumed_completed_epoch.numpy())

print("Restored completed epoch:", restored_epoch)
print("Resume begins at epoch:", restored_epoch + 1)

Restored completed epoch: 10
Resume begins at epoch: 11


In [25]:
run_training_pipeline(
    resumed_model,
    resumed_completed_epoch,
    resumed_best_epoch,
    resumed_best_validation_loss,
    train_inputs,
    train_targets,
    validation_dataset,
    TOTAL_EPOCHS,
    LEARNING_RATE,
    resumed_last_manager,
    resumed_best_manager,
)

print("\nCompleted epoch:", int(resumed_completed_epoch.numpy()))
print("Best validation epoch:", int(resumed_best_epoch.numpy()))
print("Best validation loss:", f"{float(resumed_best_validation_loss.numpy()):.6f}")

Epoch  20 | Train loss: 5.022075 | Validation loss: 5.090270

Completed epoch: 20
Best validation epoch: 18
Best validation loss: 5.088099


## 9. Restore the Best Model

After the 20-epoch budget is complete, a new model instance is restored from the checkpoint with the lowest validation loss.

Training and validation losses are recomputed using this restored `final_model`.

The test split has not influenced tokenizer construction, parameter updates or checkpoint selection. Held-out test evaluation begins only in the next section.

In [26]:
final_model = DecoderOnlyLanguageModel(vocabulary_size=vocabulary_size, seed=SEED)
_ = final_model(example_input)

selected_best_epoch = tf.Variable(0, dtype=tf.int64, trainable=False)
selected_best_validation_loss = tf.Variable(np.inf, dtype=tf.float32, trainable=False)

final_best_checkpoint = tf.train.Checkpoint(
    model=final_model,
    best_epoch=selected_best_epoch,
    best_validation_loss=selected_best_validation_loss,
)

if resumed_best_manager.latest_checkpoint is None:
    raise RuntimeError("Best checkpoint not found.")

final_best_checkpoint.restore(resumed_best_manager.latest_checkpoint).assert_existing_objects_matched()

final_train_loss = evaluate_model(final_model, make_dataset(train_inputs, train_targets, BATCH_SIZE),)

final_validation_loss = evaluate_model(final_model, validation_dataset)

print("Selected best epoch:", int(selected_best_epoch.numpy()))
print(f"Final training loss: {final_train_loss:.6f}")
print(f"Final validation loss: {final_validation_loss:.6f}")
print(f"Random baseline: {random_baseline:.6f}")

Selected best epoch: 18
Final training loss: 4.994609
Final validation loss: 5.088099
Random baseline: 7.624619


## 10. Held-Out Evaluation

Version 12 reported loss on the restored best model. Version 13 makes the held-out test evaluation more interpretable by deriving additional metrics from the same next-token logits.

**Cross-entropy** is the average negative log-probability assigned to the true target tokens.

**Perplexity** is:

`perplexity = exp(cross_entropy)`

Lower perplexity means the model is less surprised by the held-out targets.

**Top-1 accuracy** measures how often the highest-logit token is correct.

**Top-5 accuracy** measures how often the true token appears among the five highest logits.

These metrics describe token prediction only; they do not directly measure factuality, coherence or usefulness.

The uniform-vocabulary baseline from Version 12 remains available for comparison.

In [27]:
def compute_evaluation_metrics(model, dataset):
    total_negative_log_likelihood = 0.0
    total_tokens = 0
    top_1_correct = 0
    top_5_correct = 0

    for batch_inputs, batch_targets in dataset:
        _, _, batch_logits = model(batch_inputs)

        token_losses = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=batch_targets, logits=batch_logits,)

        flattened_logits = tf.reshape(batch_logits, (-1, vocabulary_size))
        flattened_targets = tf.reshape(batch_targets, (-1,))

        top_1_predictions = tf.argmax(flattened_logits, axis=-1, output_type=tf.int32,)

        top_1_correct += int(tf.reduce_sum(tf.cast(top_1_predictions == flattened_targets, tf.int32,)))

        top_5_correct += int(tf.reduce_sum(tf.cast(
            tf.math.in_top_k(
                targets=flattened_targets,
                predictions=flattened_logits,
                k=5,
            ),
            tf.int32,
        )))

        total_negative_log_likelihood += float(tf.reduce_sum(token_losses))
        total_tokens += int(tf.size(batch_targets))

    cross_entropy = total_negative_log_likelihood / total_tokens

    return {
        "cross_entropy": cross_entropy,
        "perplexity": float(np.exp(cross_entropy)),
        "top_1_accuracy": top_1_correct / total_tokens,
        "top_5_accuracy": top_5_correct / total_tokens,
        "tokens": total_tokens,
    }

In [28]:
test_metrics = compute_evaluation_metrics(final_model, test_dataset)
test_loss = test_metrics["cross_entropy"]
random_perplexity = float(np.exp(random_baseline))

print("Held-out test metrics:")
print(f"Cross-entropy: {test_metrics['cross_entropy']:.6f}")
print(f"Perplexity:    {test_metrics['perplexity']:.3f}")
print(f"Top-1 accuracy: {test_metrics['top_1_accuracy']:.4f}")
print(f"Top-5 accuracy: {test_metrics['top_5_accuracy']:.4f}")
print(f"Evaluated tokens: {test_metrics['tokens']}")

print(f"\nUniform baseline cross-entropy: {random_baseline:.6f}")
print(f"Uniform baseline perplexity: {random_perplexity:.1f}")

Held-out test metrics:
Cross-entropy: 5.105321
Perplexity:    164.897
Top-1 accuracy: 0.1030
Top-5 accuracy: 0.2402
Evaluated tokens: 32000

Uniform baseline cross-entropy: 7.624619
Uniform baseline perplexity: 2048.0


## 11. Controlled Next-Token Distributions

Version 12 used greedy decoding, selecting the highest-probability token at every step.

Version 13 keeps the logits explicit and adds three common controls before sampling:

- **temperature** rescales logits before softmax; values below 1 sharpen the distribution and values above 1 flatten it;
- **top-k** keeps only the `k` highest-probability tokens;
- **top-p** keeps the smallest descending-probability prefix whose cumulative probability reaches `p`.

After filtering, the retained probabilities are normalized again.

No high-level generation API is used.

In [29]:
def controlled_probability_distribution(logits, temperature=1.0, top_k=None, top_p=None):
    if temperature <= 0:
        raise ValueError("temperature must be greater than zero")

    logits = np.asarray(logits, dtype=np.float64)
    scaled_logits = logits / temperature
    scaled_logits -= np.max(scaled_logits)

    probabilities = np.exp(scaled_logits)
    probabilities /= probabilities.sum()

    if top_k is not None:
        if not isinstance(top_k, int) or not 1 <= top_k <= len(probabilities):
            raise ValueError("top_k must be an integer inside the vocabulary range")

        retained_indices = np.argpartition(probabilities, -top_k)[-top_k:]
        filtered = np.zeros_like(probabilities)
        filtered[retained_indices] = probabilities[retained_indices]
        probabilities = filtered / filtered.sum()

    if top_p is not None:
        if not 0 < top_p <= 1:
            raise ValueError("top_p must be in the interval (0, 1]")

        sorted_indices = np.argsort(probabilities)[::-1]
        cumulative = np.cumsum(probabilities[sorted_indices])

        retained_count = (int(np.searchsorted(cumulative, top_p, side="left")) + 1)

        retained_indices = sorted_indices[:retained_count]

        filtered = np.zeros_like(probabilities)
        filtered[retained_indices] = probabilities[retained_indices]
        probabilities = filtered / filtered.sum()

    return probabilities

In [30]:
prediction_context = test_inputs[0].numpy()

_, _, context_logits = final_model(
    tf.constant([prediction_context], dtype=tf.int32)
)

final_position_logits = context_logits[0, -1].numpy()

distribution_configurations = {
    "temperature 0.7": controlled_probability_distribution(final_position_logits, temperature=0.7,),
    "temperature 1.3": controlled_probability_distribution(final_position_logits, temperature=1.3,),
    "top-k 20": controlled_probability_distribution(final_position_logits, top_k=20,),
    "top-p 0.9": controlled_probability_distribution(final_position_logits, top_p=0.9,),
}

print("Context:")
print(tokenizer.decode(prediction_context.tolist()))

print("\nControlled distributions:")

for name, distribution in distribution_configurations.items():
    top_token_id = int(np.argmax(distribution))
    top_token = tokenizer.id_to_token(top_token_id)
    support = int(np.count_nonzero(distribution))

    print(
        f"{name:17s} | "
        f"support={support:4d} | "
        f"top={top_token!r:16s} | "
        f"p={distribution[top_token_id]:.4f}"
    )

Context:
Little Gidding is the fourth and final poem

Controlled distributions:
temperature 0.7   | support=2048 | top=',</w>'          | p=0.3661
temperature 1.3   | support=2048 | top=',</w>'          | p=0.0660
top-k 20          | support=  20 | top=',</w>'          | p=0.2788
top-p 0.9         | support= 315 | top=',</w>'          | p=0.1665


## 12. Controlled Autoregressive Generation

Generation still uses the most recent 16 BPE tokens as context.

Greedy decoding remains available as the deterministic baseline.

For stochastic decoding, Version 13 samples from the controlled probability distribution produced by temperature, top-k or top-p.

A local NumPy random generator is initialized from an explicit seed and reused through the complete generated sequence. The same model, prompt, configuration and seed therefore reproduce the same token identifiers in the same environment.

All configurations below use the same held-out prompt and generate 40 additional BPE tokens so their behavior can be compared directly.

In [31]:
def next_token_logits(model, context_ids):
    input_tensor = tf.constant([context_ids], dtype=tf.int32)
    _, _, logits = model(input_tensor)

    return logits[0, -1].numpy()


def generate_text(
    model,
    tokenizer,
    prompt,
    number_of_tokens,
    method="greedy",
    temperature=1.0,
    top_k=None,
    top_p=None,
    seed=42,
):
    generated_ids = tokenizer.encode(prompt).ids

    if len(generated_ids) < CONTEXT_LENGTH:
        raise ValueError(f"The prompt must contain at least {CONTEXT_LENGTH} tokens.")

    random_generator = np.random.default_rng(seed)

    for _ in range(number_of_tokens):
        context_ids = generated_ids[-CONTEXT_LENGTH:]
        logits = next_token_logits(model, context_ids)

        if method == "greedy":
            next_token_id = int(np.argmax(logits))

        elif method == "sample":
            probabilities = controlled_probability_distribution(
                logits,
                temperature=temperature,
                top_k=top_k,
                top_p=top_p,
            )

            next_token_id = int(random_generator.choice(vocabulary_size, p=probabilities,))

        else:
            raise ValueError("method must be 'greedy' or 'sample'")

        generated_ids.append(next_token_id)

    return generated_ids, tokenizer.decode(generated_ids)

In [32]:
generation_prompt = tokenizer.decode(prediction_context.tolist())
NUMBER_OF_GENERATED_TOKENS = 40

generation_configurations = [
    ("greedy", dict(method="greedy")),
    ("temperature 0.7", dict(method="sample", temperature=0.7)),
    ("top-k 20", dict(method="sample", top_k=20)),
    ("top-p 0.9", dict(method="sample", top_p=0.9)),
]

generation_results = {}

print("Prompt:")
print(generation_prompt)

for name, configuration in generation_configurations:
    generated_ids, generated_text = generate_text(
        final_model,
        tokenizer,
        generation_prompt,
        NUMBER_OF_GENERATED_TOKENS,
        seed=SEED,
        **configuration,
    )

    generation_results[name] = {"ids": generated_ids, "text": generated_text,}

    print(f"\n{name}:")
    print(generated_text)

Prompt:
Little Gidding is the fourth and final poem

greedy:
Little Gidding is the fourth and final poem , and the Sedla , and the See , and the See , and the Sedla , and the See , and the See

temperature 0.7:
Little Gidding is the fourth and final poem and the chisumpoints in a , and engen an ' s and Heximan Wellelcisomekurted any , and the Nem . The

top-k 20:
Little Gidding is the fourth and final poem of " and the City in the CC . It is in the DC Holually , who also had the Pakoned his Trand Hevoy

top-p 0.9:
Little Gidding is the fourth and final poem is hiway is healents by Lenm Compogual gemeks he obinsted two time , in Emusetne , and of Smacually


## 13. Inspecting Attention

Controlled decoding changes how a token is selected from the final-position logits; it does not change the Transformer attention calculation.

The restored best model still produces one causal `16 × 16` attention matrix per Transformer block.

The following inspection uses the same `final_model` selected by validation and used for held-out evaluation and generation.

In [33]:
attention_context = test_inputs[0:1]

_, attention_weights, _ = final_model(attention_context)

attention_tokens = [
    tokenizer.id_to_token(int(token_id))
    for token_id in attention_context[0].numpy()
]

print("Context:")
print(tokenizer.decode(attention_context[0].numpy().tolist()))

print("\nTokens:")
print(attention_tokens)

Context:
Little Gidding is the fourth and final poem

Tokens:
['L', 'it', 'tle</w>', 'G', 'id', 'ding</w>', 'is</w>', 'the</w>', 'f', 'our', 'th</w>', 'and</w>', 'final</w>', 'po', 'e', 'm</w>']


In [34]:
for block_index, block_attention in enumerate(attention_weights, start=1):
    final_position_attention = block_attention[0, -1].numpy()

    print(f"\nTransformer block {block_index}")
    print("-" * 45)

    for token, weight in zip(attention_tokens, final_position_attention,):
        print(f"{token!r:18s} {weight:.4f}")

    print("Attention sum:", f"{final_position_attention.sum():.6f}",)


Transformer block 1
---------------------------------------------
'L'                0.0625
'it'               0.0625
'tle</w>'          0.0625
'G'                0.0624
'id'               0.0625
'ding</w>'         0.0625
'is</w>'           0.0625
'the</w>'          0.0627
'f'                0.0624
'our'              0.0625
'th</w>'           0.0625
'and</w>'          0.0626
'final</w>'        0.0625
'po'               0.0625
'e'                0.0625
'm</w>'            0.0625
Attention sum: 1.000000

Transformer block 2
---------------------------------------------
'L'                0.1064
'it'               0.0581
'tle</w>'          0.0128
'G'                0.3096
'id'               0.0204
'ding</w>'         0.0059
'is</w>'           0.0051
'the</w>'          0.0295
'f'                0.3502
'our'              0.0359
'th</w>'           0.0118
'and</w>'          0.0133
'final</w>'        0.0083
'po'               0.0075
'e'                0.0058
'm</w>'            0.0193
Attention 

## 14. Results

Version 13 preserves the complete Version 12 training and checkpoint pipeline and extends only the post-training stage.

The restored best model is evaluated on the held-out test split using cross-entropy, perplexity, top-1 accuracy and top-5 accuracy.

The same final-position logits are then transformed explicitly with temperature, top-k and top-p before sampling.

Autoregressive generation compares greedy decoding with three controlled sampling configurations using the same prompt, generation length and random seed.

The Version 13 progression is therefore:

Version 12 restored best model  
→ held-out evaluation  
→ explicit decoding controls  
→ reproducible controlled generation

The model architecture, tokenizer, data selection and training budget remain unchanged.

## 15. Tests

The following checks preserve the structural and checkpoint guarantees of Version 12 and verify the new Version 13 behavior.

They confirm that held-out metrics are finite and internally consistent, perplexity matches the exponential of cross-entropy, top-5 accuracy is at least top-1 accuracy, controlled distributions remain normalized, top-k retains the configured support, invalid temperatures are rejected, seeded sampling is reproducible, generation adds the expected number of tokens, and causal attention remains correctly masked.

In [35]:
assert vocabulary_size == VOCABULARY_SIZE
assert tokenizer.token_to_id("[UNK]") is not None

assert train_inputs.shape == (MAX_TRAIN_SEQUENCES, CONTEXT_LENGTH)
assert train_targets.shape == (MAX_TRAIN_SEQUENCES, CONTEXT_LENGTH)
assert validation_inputs.shape == (MAX_VALIDATION_SEQUENCES, CONTEXT_LENGTH)
assert validation_targets.shape == (MAX_VALIDATION_SEQUENCES, CONTEXT_LENGTH)
assert test_inputs.shape == (MAX_TEST_SEQUENCES, CONTEXT_LENGTH)
assert test_targets.shape == (MAX_TEST_SEQUENCES, CONTEXT_LENGTH)

assert np.array_equal(train_inputs[0, 1:].numpy(), train_targets[0, :-1].numpy())

assert model_output.shape == (1, CONTEXT_LENGTH, EMBEDDING_DIM)
assert logits.shape == (1, CONTEXT_LENGTH, vocabulary_size)

assert len(model.blocks) == NUM_BLOCKS
assert model.blocks[0] is not model.blocks[1]
assert len(model.trainable_variables) == 23
assert total_parameters == 34496

assert last_manager.latest_checkpoint is not None
assert best_manager.latest_checkpoint is not None
assert resumed_last_manager.latest_checkpoint is not None
assert resumed_best_manager.latest_checkpoint is not None

assert restored_epoch == INTERRUPT_AFTER_EPOCH
assert int(resumed_completed_epoch.numpy()) == TOTAL_EPOCHS
assert int(selected_best_epoch.numpy()) == int(resumed_best_epoch.numpy())

assert np.isclose(float(selected_best_validation_loss.numpy()), float(resumed_best_validation_loss.numpy()), atol=1e-6)

loss_values = [initial_train_loss, initial_validation_loss, final_train_loss, final_validation_loss, test_loss]

assert np.all(np.isfinite(loss_values))
assert final_train_loss < initial_train_loss
assert final_validation_loss < initial_validation_loss
assert test_loss < random_baseline

assert np.isclose(final_validation_loss, float(selected_best_validation_loss.numpy()), atol=1e-5)

assert np.isfinite(test_metrics["perplexity"])
assert np.isclose(test_metrics["perplexity"], np.exp(test_metrics["cross_entropy"]))
assert test_metrics["tokens"] == MAX_TEST_SEQUENCES * CONTEXT_LENGTH
assert 0 <= test_metrics["top_1_accuracy"] <= test_metrics["top_5_accuracy"] <= 1

assert np.isclose(random_perplexity, vocabulary_size)

for distribution in distribution_configurations.values():
    assert np.all(np.isfinite(distribution))
    assert np.all(distribution >= 0)
    assert np.isclose(distribution.sum(), 1.0)

assert np.count_nonzero(distribution_configurations["top-k 20"]) == 20

try:
    controlled_probability_distribution(final_position_logits, temperature=0)
except ValueError:
    pass
else:
    raise AssertionError("Non-positive temperature was not rejected.")

prompt_length = len(tokenizer.encode(generation_prompt).ids)

for result in generation_results.values():
    assert len(result["ids"]) == prompt_length + NUMBER_OF_GENERATED_TOKENS
    assert min(result["ids"]) >= 0
    assert max(result["ids"]) < vocabulary_size

repeated_top_p_ids, _ = generate_text(
    final_model,
    tokenizer,
    generation_prompt,
    NUMBER_OF_GENERATED_TOKENS,
    method="sample",
    top_p=0.9,
    seed=SEED,
)

assert repeated_top_p_ids == generation_results["top-p 0.9"]["ids"]

for block_attention in attention_weights:
    matrix = block_attention[0].numpy()
    assert matrix.shape == (CONTEXT_LENGTH, CONTEXT_LENGTH)
    assert np.allclose(matrix.sum(axis=-1), 1.0, atol=1e-6)
    assert np.allclose(np.triu(matrix, k=1), 0.0, atol=1e-7)

print("All checks passed.")

All checks passed.


## Notes

- Version 13 is Version 12 plus held-out evaluation and controlled generation.

- WikiText-103 Raw v1, BPE tokenization, sequence limits, model dimensions, checkpoint behavior, learning rate, batch size and 20-epoch training budget remain unchanged.

- The best validation checkpoint is restored into `final_model` before the test split is evaluated.

- Test cross-entropy is complemented by perplexity, top-1 accuracy and top-5 accuracy. The uniform-vocabulary baseline remains the simple reference used in Version 12.

- Temperature, top-k and top-p are implemented directly from the final-position logits rather than through a high-level generation API.

- Stochastic generation uses a local seeded NumPy random generator, making repeated sampling reproducible for the same model, prompt, controls and seed.

- Every generation configuration uses the same prompt and adds 40 BPE tokens so decoding behavior can be compared directly.

- The runtime checkpoint directory is recreated at the beginning of each execution. Persistent checkpoint storage across separate Kaggle sessions remains outside the scope of this educational version.

- Generated text remains limited by the deliberately small model, short context window, restricted training subset and simple decoding controls.

- Because whitespace pre-tokenization treats punctuation as separate units, decoded text may contain spaces before punctuation.

Future versions will introduce new components and gradually evolve the architecture.